In [9]:
from pathlib import Path
import re
from collections import defaultdict

import numpy as np
import pandas as pd

EXP = "debug"
N_LEVELS = 4
RIGID = "Rigid"
AFFINE = "Affine"
SYN = "SyN"
STAGES = [RIGID, AFFINE, SYN]
# STAGES = ["Rigid", "Affine"]  # SyN is not working right now

p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST.*|  Elapsed time"
log = Path(f"log-{EXP}.txt").read_text()
raw_data = [
    [y.strip() for y in x.split("\n") if y.strip() not in ["", "XX"]]
    for i, x in enumerate(re.split(p_lvl_header, log))
    if i % 5 != 0
]

In [10]:
dfs = list()
for stage in STAGES:
    for level in range(N_LEVELS):
        table = defaultdict(list)
        for line in raw_data[level + N_LEVELS * STAGES.index(stage)]:
            if not (line.startswith("2DIAGNOSTIC") or line.startswith("1DIAGNOSTIC")):
                # print(f"Skipping line: {line}")
                continue

            cols = line.split(",")
            table["subject_id"] = "007"  # TODO retrieve dynamically
            table["iteration"].append(int(cols[1]))
            table["metricValue"].append(np.float64(cols[2]))

            if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
                table["lipschitz"].append(np.float64(cols[6]))
                table["params_2norm"].append(np.float64(cols[7]))
                table["grad_2norm"].append(np.float64(cols[8]))

            elif line.startswith("1DIAGNOSTIC"):  # SyN
                # Moving
                table["moving_lipschitz"].append(np.float64(cols[6]))
                table["moving_params_2norm"].append(np.float64(cols[7]))
                table["moving_grad_2norm"].append(np.float64(cols[8]))
                # Fixed
                table["fixed_lipschitz"].append(np.float64(cols[9]))
                table["fixed_params_2norm"].append(np.float64(cols[10]))
                table["fixed_grad_2norm"].append(np.float64(cols[11]))

        common_data = {
            "exp": EXP,
            "stage": stage,
            "level": level,
            "subject_id": table["subject_id"],
            "iteration": table["iteration"],
            "metricValue": table["metricValue"],
        }

        if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
            dfs.append(
                pd.DataFrame(
                    data=common_data
                    | {
                        "lipschitz": table["lipschitz"],
                        "params_2norm": table["params_2norm"],
                        "grad_2norm": table["grad_2norm"],
                    }
                )
            )
        elif line.startswith("1DIAGNOSTIC"):  # SyN
            # Moving
            dfs.append(
                pd.DataFrame(
                    data=common_data
                    | {
                        "direction": "moving",
                        "lipschitz": table["moving_lipschitz"],
                        "params_2norm": table["moving_params_2norm"],
                        "grad_2norm": table["moving_grad_2norm"],
                    }
                )
            )
            # Fixed
            dfs.append(
                pd.DataFrame(
                    data=common_data
                    | {
                        "direction": "fixed",
                        "lipschitz": table["fixed_lipschitz"],
                        "params_2norm": table["fixed_params_2norm"],
                        "grad_2norm": table["fixed_grad_2norm"],
                    }
                )
            )

df = pd.concat(dfs, ignore_index=True)
df = df.astype(
    {
        "exp": str,
        "stage": str,
        "level": int,
        "iteration": int,
        "subject_id": str,
        "direction": str,
        "lipschitz": np.float64,
        "params_2norm": np.float64,
        "grad_2norm": np.float64,
        "metricValue": np.float64,
    }
)
df["pmin"] = np.log2(
    (4 + 3 * np.sqrt(2)) / df["grad_2norm"] * df["lipschitz"] * df["params_2norm"]
)


# Simulating multiple subjects by adding noise to pmin
rng = np.random.default_rng(123)

fake_df = df.copy()
fake_df["pmin"] = fake_df["pmin"] + rng.normal(loc=fake_df["pmin"], scale=3)
fake_df["subject_id"] = "008"
# df = pd.concat([df, fake_df], ignore_index=True)

In [11]:
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    r, g, b = hex_to_rgb(hex_color)
    return f"rgba({r}, {g}, {b}, {alpha})"


def plot_error(
    fig: go.Figure,
    df: pd.DataFrame,
    *,
    row: int,
    col: int,
    name: str,
    color: str,
    dash: str = "solid",
    metric: str = "pmin",
    showlegend: bool = False,
):
    BOUNDARY_WIDTH = 0.5
    # Upper bound
    fig.add_trace(
        go.Scatter(
            x=df["iteration"],
            y=df.groupby("iteration")[metric].max(),
            mode="lines",
            line=dict(color=color, width=BOUNDARY_WIDTH, dash=dash),
            showlegend=False,
            name="Upper Bound",
        ),
        row=row,
        col=col,
    )

    # Lower bound with fill to upper
    fig.add_trace(
        go.Scatter(
            x=df["iteration"],
            y=df.groupby("iteration")[metric].min(),
            fill="tonexty",  # fill area between this and previous trace
            fillcolor=hex_to_rgba(color, 0.2),
            mode="lines",
            line=dict(color=color, width=BOUNDARY_WIDTH, dash=dash),
            showlegend=False,
            name="Lower Bound",
        ),
        row=row,
        col=col,
    )

    # Mean line
    fig.add_trace(
        go.Scatter(
            x=df["iteration"],
            y=df.groupby("iteration")[metric].mean(),
            mode="lines",
            line=dict(color=color, width=2, dash=dash),
            name=name,
            showlegend=showlegend,
        ),
        row=row,
        col=col,
    )


In [39]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

METRIC = "pmin"

BOKEH_COLORBLIND = (
    "#0072B2",  # 0
    "#E69F00",  # 1
    "#F0E442",  # 2
    "#009E73",  # 3
    "#56B4E9",  # 4
    "#D55E00",  # 5
    "#CC79A7",  # 6
    "#000000",  # 7
)
COLORS = {
    "Rigid": BOKEH_COLORBLIND[3],
    "Affine": BOKEH_COLORBLIND[6],
    "SyN (Moving)": BOKEH_COLORBLIND[0],
    "SyN (Fixed)": BOKEH_COLORBLIND[1],
}
fig_dir = Path("figures")


def add_hline(fig, y, text):
    # Only show the horizontal line in the last column
    fig.add_hline(
        y,
        line=dict(dash="dot"),
    )
    fig.add_hline(
        y,
        col=N_LEVELS,
        line=dict(dash="dot"),
        annotation_text=text,
        annotation_position="right",
        annotation_xref="paper",
        annotation_font_size=12,
        annotation_showarrow=False,
        annotation_font_color="black",
    )


# Get sorted unique stages and levels
stages = df["stage"].unique()
levels = sorted(df["level"].unique())

# Create subplot figure
fig = make_subplots(
    rows=len(STAGES),
    cols=N_LEVELS,
    subplot_titles=[
        f"Stage: {stage} | Level: {level}" for stage in stages for level in levels
    ],
    shared_xaxes=False,
    shared_yaxes=True,
    horizontal_spacing=0.025,
    vertical_spacing=0.1,
)
# Update subplot titles font size
for annotation in fig.layout.annotations:
    annotation.font = dict(size=18)

# Map stage to row index
stage_to_row = {stage: i + 1 for i, stage in enumerate(stages)}
level_to_col = {level: i + 1 for i, level in enumerate(levels)}

# Add traces
for i, stage in enumerate(stages):
    for j, level in enumerate(levels):
        sub_df = df[(df["stage"] == stage) & (df["level"] == level)].copy()
        row = stage_to_row[stage]
        col = level_to_col[level]

        showlegend = True if j == 0 else False
        if stage in [RIGID, AFFINE]:
            plot_error(
                fig,
                sub_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name=stage,
                color=COLORS[stage],
            )
        elif stage == SYN:
            moving_df = sub_df[sub_df["direction"] == "moving"]
            fixed_df = sub_df[sub_df["direction"] == "fixed"]

            # Moving
            plot_error(
                fig,
                moving_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name="SyN (Moving)",
                color=COLORS["SyN (Moving)"],
                dash="solid",
            )

            # Fixed
            plot_error(
                fig,
                fixed_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name="SyN (Fixed)",
                color=COLORS["SyN (Fixed)"],
                dash="solid",
            )

# Only add this ONCE (e.g., at subplot [1,1] or outside the loop)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[0],
        mode="lines",
        line=dict(dash="dot", color="black"),
        name="hardware support",
        showlegend=True,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)
add_hline(fig, y=53, text="FP64")
add_hline(fig, y=24, text="FP32")
add_hline(fig, y=16, text="FP24")
add_hline(fig, y=11, text="TF32")
add_hline(fig, y=8, text="BF16")

# Layout adjustments
fig.update_layout(
    height=len(STAGES) * 400,
    width=1200,
    font=dict(
        size=24,
        color="black",
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.025,  # position above the subplots
        xanchor="left",
        x=0,
        traceorder="normal",
    ),
    title="Estimated minimum bit precision at each iteration",
    margin=dict(l=10, r=40, t=150, b=10),
    plot_bgcolor="white",
)
for i in range(1, len(STAGES) * N_LEVELS + 1):
    fig.update_layout(
        {
            f"xaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
            f"yaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
        }
    )

# Set x-axis and y-axis titles
for i, _ in enumerate(levels):
    fig.update_xaxes(title_text="nth Iteration", row=len(STAGES), col=i + 1)

fig.show()
# Save the figure
fig.update_layout(
    title=None,
    margin=dict(l=10, r=40, t=10, b=10),
)
fig.write_image(
    fig_dir / f"pmin.pdf",
    width=1200,
    height=len(STAGES) * 400,
    scale=2,
)


In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

METRIC = "lipschitz"
BOKEH_COLORBLIND = (
    "#0072B2",  # 0
    "#E69F00",  # 1
    "#F0E442",  # 2
    "#009E73",  # 3
    "#56B4E9",  # 4
    "#D55E00",  # 5
    "#CC79A7",  # 6
    "#000000",  # 7
)
COLORS = {
    "Rigid": BOKEH_COLORBLIND[3],
    "Affine": BOKEH_COLORBLIND[6],
    "SyN (Moving)": BOKEH_COLORBLIND[0],
    "SyN (Fixed)": BOKEH_COLORBLIND[1],
}
fig_dir = Path("figures")


def add_hline(fig, y, text):
    # Only show the horizontal line in the last column
    fig.add_hline(
        y,
        line=dict(dash="dot"),
    )
    fig.add_hline(
        y,
        col=N_LEVELS,
        line=dict(dash="dot"),
        annotation_text=text,
        annotation_font_size=20,
        annotation_showarrow=False,
        annotation_font_color="black",
    )


# Get sorted unique stages and levels
stages = df["stage"].unique()
levels = sorted(df["level"].unique())

# Create subplot figure
fig = make_subplots(
    rows=len(STAGES),
    cols=N_LEVELS,
    subplot_titles=[
        f"Stage: {stage} | Level: {level}" for stage in stages for level in levels
    ],
    shared_xaxes=False,
    shared_yaxes=True,
    horizontal_spacing=0.025,
    vertical_spacing=0.1,
)
# Update subplot titles font size
for annotation in fig.layout.annotations:
    annotation.font = dict(size=18)

# Map stage to row index
stage_to_row = {stage: i + 1 for i, stage in enumerate(stages)}
level_to_col = {level: i + 1 for i, level in enumerate(levels)}

# Add traces
for i, stage in enumerate(stages):
    for j, level in enumerate(levels):
        sub_df = df[(df["stage"] == stage) & (df["level"] == level)].copy()
        row = stage_to_row[stage]
        col = level_to_col[level]

        showlegend = True if j == 0 else False
        if stage in [RIGID, AFFINE]:
            plot_error(
                fig,
                sub_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name=stage,
                color=COLORS[stage],
            )
        elif stage == SYN:
            moving_df = sub_df[sub_df["direction"] == "moving"]
            fixed_df = sub_df[sub_df["direction"] == "fixed"]

            # Moving
            plot_error(
                fig,
                moving_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name="SyN (Moving)",
                color=COLORS["SyN (Moving)"],
                dash="solid",
            )

            # Fixed
            plot_error(
                fig,
                fixed_df,
                row=row,
                col=col,
                metric=METRIC,
                showlegend=showlegend,
                name="SyN (Fixed)",
                color=COLORS["SyN (Fixed)"],
                dash="solid",
            )

# Only add this ONCE (e.g., at subplot [1,1] or outside the loop)
fig.add_trace(
    go.Scatter(
        x=[1],
        y=[0],
        mode="lines",
        line=dict(dash="dot", color="black"),
        name="hardware support",
        showlegend=True,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)
# add_hline(fig, y=53, text="FP64")
# add_hline(fig, y=24, text="FP32")
# add_hline(fig, y=16, text="FP24")
# add_hline(fig, y=11, text="TF32")
# add_hline(fig, y=8, text="BF16")

# Layout adjustments
fig.update_layout(
    height=len(STAGES) * 400,
    width=1200,
    font=dict(
        size=24,
        color="black",
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.025,  # position above the subplots
        xanchor="left",
        x=0,
        traceorder="normal",
    ),
    title=f"Metric ({METRIC}) each iteration",
    margin=dict(l=10, r=10, t=150, b=10),
    plot_bgcolor="white",
)
for i in range(1, len(STAGES) * N_LEVELS + 1):
    fig.update_layout(
        {
            f"xaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
            f"yaxis{i}": dict(
                showline=True,
                linewidth=2,
                linecolor="black",
                type="log",
                dtick=1,
            ),
        }
    )

# Set x-axis and y-axis titles
for i, _ in enumerate(levels):
    fig.update_xaxes(title_text="Iteration", row=len(STAGES), col=i + 1)

fig.show()
